# A File Split Into Many Trees

**Version**: 1

**Last modified**: 2026-10-12

**Compatible with opengate-gate-tree version:** 0.5.0

**Author**: Mateusz Jakub Bała

## About

GATE can write the hits of one simulation into several trees of one file: a
new tree for every run after the first, or one tree per sensitive detector.
Each of them is a whole "Hits" tree of the same structure, and an analysis
usually wants them together.

This notebook reads both kinds of split file. It shows what the package does
when it is asked for "the hits" of such a file without being told which tree,
how to read every tree as one dataset, and why the merged dataset carries a
column naming the tree each row came from — a column that, for a file split per
detector, is the only thing left that says where a deposit was recorded.

## What will you learn from this notebook

- how to list the trees of a file that hold hits, whatever they are called
- what happens when one tree of several is read, and when none can be chosen
- how to read one particular tree by its name
- how to read every tree as one dataset, and where each row came from
- why the same event turns up in two trees of a file split per detector
- how to merge trees read separately, and what merging refuses

## Tutorial

Every step below is a line of text, the code that does what the line says, and
a word about the answer where the answer needs one.

In [1]:
from tools import environment, paths

print(environment.describe())

opengate-gate-tree: 0.5.0
python: 3.14.5
run on: 2026-10-12


The header of this notebook states the version it was written for; the line
above states the version it just ran against. If they differ, the notebook may
still be right — but nobody has checked it against what is installed.

The two files this notebook reads, both described in
[`../data/README.md`](../data/README.md): one split per run, one split per
sensitive detector. Every tree in them is cut to its first 500 entries.

The messages below name the file they are about, so the files are named
relative to this notebook — the way a reader sees them in the repository —
rather than by the full path on the machine that ran it.

In [2]:
import os
from pathlib import Path

per_run = Path(os.path.relpath(paths.input_example("hits-one-tree-per-run.root")))
per_detector = Path(os.path.relpath(paths.input_example("hits-one-tree-per-detector.root")))
per_run, per_detector

(PosixPath('../data/input-examples/hits-one-tree-per-run.root'),
 PosixPath('../data/input-examples/hits-one-tree-per-detector.root'))

In [3]:
from opengate_gate_tree import (
    SOURCE_TREE_BRANCH,
    AmbiguousTreeError,
    GateTree,
    GateTreeError,
    RootFile,
    merge_tree_data,
    read_hits_trees,
    read_tree,
)

The package reports what it decides on its own through its log, which is
silent until an application gives it somewhere to write. Here that is a handler
printing the level and the message:

In [4]:
import logging
import sys

from opengate_gate_tree import LOGGER_NAME

handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(logging.Formatter("%(levelname)s: %(message)s"))
logging.getLogger(LOGGER_NAME).addHandler(handler)

### One Tree Per Run

The trees of the file, and the ones among them that hold hits.
`hits_tree_names()` recognises a tree by its branches rather than by its name,
so it finds hits stored under any name:

In [5]:
with RootFile(per_run) as root_file:
    print(root_file.tree_names)
    print(root_file.hits_tree_names())

('Hits', 'Hits_run1', 'Hits_run2')
('Hits', 'Hits_run1', 'Hits_run2')


GATE names the tree of the first run `Hits` and those of the runs after it
`Hits_run1`, `Hits_run2`. Asking for the "Hits" tree finds one of that name, so
that is the tree read — and a warning names the two that were left out:

In [6]:
first_run = read_tree(per_run, GateTree.HITS, ["runID", "eventID"])
first_run

TreeData(tree='Hits', entries=500, branches=2)

Reading one run out of three without a word would be the worst outcome
available, so the package says what it skipped. One particular tree is read by
naming it:

In [7]:
second_run = read_tree(per_run, GateTree.HITS, ["runID", "eventID"], tree_name="Hits_run1")
set(second_run["runID"].tolist())

{1}

### Every Tree As One Dataset

`read_hits_trees` reads every tree holding hits, in file order, and places
their rows one after another:

In [8]:
runs = read_hits_trees(per_run, ["runID", "eventID", "trackID", "edep"])
runs

TreeData(tree='Hits', entries=1500, branches=5)

Four branches were asked for and five came back: the added one is
`sourceTreeName`, under the name `SOURCE_TREE_BRANCH`, and it names the tree
every row was read from. Against `runID`, it says what one would expect of a
file split per run — each tree holds one run:

In [9]:
import pandas as pd

runs_frame = runs.to_dataframe()
pd.crosstab(runs_frame[SOURCE_TREE_BRANCH], runs_frame["runID"])

runID,0,1,2
sourceTreeName,,,
Hits,500,0,0
Hits_run1,0,500,0
Hits_run2,0,0,500


Here the column repeats what `runID` already says. In the next file it does
not.

### One Tree Per Detector

A simulation with two sensitive detectors can write one tree for each. Neither
is named `Hits`, and choosing a detector is not something the package can do
for an analysis — so it refuses, and says what the file holds:

In [10]:
try:
    read_tree(per_detector, GateTree.HITS)
except AmbiguousTreeError as error:
    print(f"{type(error).__name__}: {error}")

AmbiguousTreeError: File ../data/input-examples/hits-one-tree-per-detector.root holds hits in 2 trees: ['Hits_DET_INNER', 'Hits_DET_OUTER']. None of them is named 'Hits', so the one to read cannot be chosen. Name the tree to read, or read them all together as one dataset.


Both detectors together, with the tree each row came from:

In [11]:
detectors = read_hits_trees(
    per_detector, ["runID", "eventID", "trackID", "edep", "posX", "posY", "posZ"]
)
detectors_frame = detectors.to_dataframe()

detectors_frame.groupby(SOURCE_TREE_BRANCH).agg(
    hits=("eventID", "size"),
    events=("eventID", "nunique"),
    runs=("runID", "nunique"),
)

,hits,events,runs
sourceTreeName,,,
Hits_DET_INNER,500,208,1
Hits_DET_OUTER,500,231,1


Both trees hold the same single run. So the same decay can be recorded in both,
under the same `runID` and the same `eventID` — one gamma stopping in the inner
detector, another in the outer one. How many events turn up in both trees:

In [12]:
inner = detectors_frame[detectors_frame[SOURCE_TREE_BRANCH] == "Hits_DET_INNER"]
outer = detectors_frame[detectors_frame[SOURCE_TREE_BRANCH] == "Hits_DET_OUTER"]

in_both = sorted(set(inner["eventID"]) & set(outer["eventID"]))
len(in_both), in_both[:5]

(134, [1, 2, 3, 5, 8])

134 events. The trees were cut to 500 entries each, and the outer one reaches
further into the run before it is cut, so the number describes these two files
rather than the simulation — but the first of them already shows what happens:

In [13]:
detectors_frame.gate.by_event(0, in_both[0])

,runID,eventID,trackID,edep,posX,posY,posZ,sourceTreeName
2,0,1,1,0.511,-52.616787,-63.610943,192.308456,Hits_DET_INNER
500,0,1,2,0.511,53.008038,64.083946,-193.738449,Hits_DET_OUTER


Event 1 is one decay: two gammas of 511 keV, flying off in opposite directions
— the positions are nearly the mirror image of each other — and each deposited
in a different detector. Once the rows are together, the identifiers say they
belong to one event, which is true, and nothing but `sourceTreeName` says which
detector recorded which deposit. That is why the column is added, and why
merging does not collapse rows sharing a run and an event: those rows are the
reason to merge in the first place.

### Merging Trees Read Separately

`read_hits_trees` reads and merges in one call. The merge alone is
`merge_tree_data`, for trees read one by one — with a selection of their own,
for instance — given in the order their rows should follow, with the name of
each:

In [14]:
branches = ["runID", "eventID", "edep"]

with RootFile(per_detector) as root_file:
    inner_tree = root_file.read(GateTree.HITS, branches, tree_name="Hits_DET_INNER")
    outer_tree = root_file.read(GateTree.HITS, branches, tree_name="Hits_DET_OUTER")

merged = merge_tree_data([inner_tree, outer_tree], ["Hits_DET_INNER", "Hits_DET_OUTER"])
merged, merged.branch_names

(TreeData(tree='Hits', entries=1000, branches=4),
 ('runID', 'eventID', 'edep', 'sourceTreeName'))

Merging is a concatenation and nothing else, so it refuses trees whose rows
would not line up: other branches, the same branches in another order, or a
branch stored with another type. Trees read with different selections are the
everyday way to get there:

In [15]:
with RootFile(per_detector) as root_file:
    narrow = root_file.read(GateTree.HITS, ["runID", "eventID"], tree_name="Hits_DET_OUTER")

try:
    merge_tree_data([inner_tree, narrow], ["Hits_DET_INNER", "Hits_DET_OUTER"])
except GateTreeError as error:
    print(f"{type(error).__name__}: {error}")

TreeMergeError: Tree 2 holds other branches than the first one. Missing from it: ['edep']. Held only by it: [].


When the result has to hold the branches of the structure and nothing more,
the source column is left out — at the cost of the one thing that told the two
detectors apart:

In [16]:
read_hits_trees(per_detector, branches, add_source_branch=False).branch_names

('runID', 'eventID', 'edep')

Every tree is held in memory before the trees are joined, so merging needs room
for the whole file. For a file too large for that, read the trees one at a
time by name.

## Summary

A file whose hits are split over several trees is read tree by tree with
`tree_name`, or all at once with `read_hits_trees`, which adds a
`sourceTreeName` column naming the tree of every row. The package warns when it
reads one tree of several and refuses to choose between trees none of which is
called `Hits`; in a file split per detector, that added column is the only
thing that says which detector recorded a deposit.

## Bibliography

- [Merging Trees](https://opengate-gate-tree.readthedocs.io/en/latest/guide/merging.html)
  — reading a split file, what the package does on its own, and what merging
  does not do
- [Event Identifiers](https://opengate-gate-tree.readthedocs.io/en/latest/guide/events.html)
  — why an event is a run and an event together
- [Loading files](https://opengate-gate-tree.readthedocs.io/en/latest/guide/loading.html)
  — how a tree is found
- [Command line](https://opengate-gate-tree.readthedocs.io/en/latest/guide/cli.html)
  — `--merge-hits-trees` and `--input-tree-name`

## Changelog

- v1: written.